# load packages

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from datetime import datetime

# read in input files

In [ ]:
!dx download --overwrite :/AD_GWAS/input/*

In [ ]:
phecode = pd.read_csv('phecodeX_UKBB.csv', low_memory = False, usecols = ['id', 'NS_328.11'])
phecode.head()

In [ ]:
demo = pd.read_csv('all_participants.sex.month_year_birth.2025-02-12_participant.csv', low_memory = False)
demo.head()

In [ ]:
first_date = pd.read_csv('AD_G30_Date_First_Reported_participant.csv')
first_date.head()

# filter phecode file to cases

In [ ]:
phecode['NS_328.11'].unique()

In [ ]:
case = phecode[phecode['NS_328.11'].isin([True])]
print(len(case.index))
print(case['NS_328.11'].unique())
case.head()

In [ ]:
case['AD'] = 1
case.drop(columns = ['NS_328.11'], inplace = True)
case.head()

# add first date column

In [ ]:
first_date['p131036'].unique()

In [ ]:
first_date.rename(columns = {'p131036' : 'FIRST_DATE', 'eid' : 'id'}, inplace = True)
first_date.head()

In [ ]:
case_first_date = case.merge(first_date, on = 'id')
case_first_date.dropna(inplace = True)
case_first_date['FIRST_DATE'] = pd.to_datetime(case_first_date['FIRST_DATE'])
print(case_first_date['FIRST_DATE'].dtype)
print(len(case_first_date.index))
case_first_date.head()

# filter phecode dataframe to controls

In [ ]:
control = phecode[phecode['NS_328.11'].isin([False])]
print(len(control.index))
print(control['NS_328.11'].unique())
control.head()

In [ ]:
control['AD']=0
control.drop(columns = ['NS_328.11'], inplace = True)
control.head()

# merge cases and controls back together

In [ ]:
case_control = pd.concat([case_first_date, control], axis = 0)
print(len(case_control.index))
case_control.head()

# clean demo df

## rename columns

In [ ]:
demo.rename(columns = {'eid' : 'id',
                       'p31' : 'SEX',
                       'p34' : 'YEAR_BIRTH',
                       'p52' : 'MONTH_BIRTH'}, inplace = True)
demo.head()

## recode sex column

In [ ]:
demo['SEX'].unique()

In [ ]:
demo['SEX'] = demo['SEX'].str.replace('Male', '1')
demo['SEX'] = demo['SEX'].str.replace('Female', '2')
print(demo['SEX'].unique())

## recode month column

In [ ]:
demo['MONTH_BIRTH'] = pd.to_datetime(demo['MONTH_BIRTH'], format = '%B').dt.month
demo.head()

## make date column

In [ ]:
demo['DOB'] = demo['YEAR_BIRTH'].astype(str) + '-' + demo['MONTH_BIRTH'].astype(str) + '-1'
demo.head()

In [ ]:
demo['DOB'] = pd.to_datetime(demo['DOB'], format = "%Y-%m-%d")
print(demo['DOB'].dtype)
demo.head()

## drop columns

In [ ]:
demo.drop(columns = ['MONTH_BIRTH', 'YEAR_BIRTH'], inplace = True)
print(len(demo.index))
demo.head()

# merge demo and case/control

In [ ]:
case_control_demo = case_control.merge(demo, on = 'id')
print(len(case_control_demo.index))
print(case_control_demo['AD'].value_counts())
case_control_demo.head()

# change ID column to int type

In [ ]:
case_control_demo['id'] = case_control_demo['id'].astype(int)
case_control_demo.head()

# calculate age

## cases

In [ ]:
case_demo = case_control_demo[case_control_demo['AD'].isin([1])]
case_demo['AGE'] = case_demo['FIRST_DATE'] - case_demo['DOB']
case_demo['AGE'] = case_demo['AGE'].astype(str).str.replace(' days', '')
case_demo['AGE'] = case_demo['AGE'].astype(float)
case_demo['AGE'] = case_demo['AGE']/365.2425
print(case_demo['AD'].unique())
print(len(case_demo.index))
case_demo.head()

## controls

In [ ]:
control_demo = case_control_demo[case_control_demo['AD'].isin([0])]
control_demo['AGE'] = (datetime(2023, 10, 1, 0, 0, 0)) - control_demo['DOB']
control_demo['AGE'] = control_demo['AGE'].astype(str).str.replace(' days', '')
control_demo['AGE'] = control_demo['AGE'].astype(float)
control_demo['AGE'] = control_demo['AGE']/365.2425
print(control_demo['AD'].unique())
print(len(control_demo.index))
control_demo.head()

## recombine

In [ ]:
case_control_age = pd.concat([case_demo, control_demo], axis = 0)
print(len(case_control_age.index))
case_control_age.head()

## filter to age > 65

In [ ]:
case_control_age_65 = case_control_age[case_control_age['AGE'] >= 65]
case_control_age_65.drop(columns = ['FIRST_DATE', 'DOB'], inplace = True)
print(len(case_control_age_65.index))
print(case_control_age_65['AD'].value_counts())
case_control_age_65.head()

# make sample list

In [ ]:
sample = case_control_demo[['id']]
sample.head()

# export sample list

In [ ]:
sample.to_csv('UKBB.AD.ALL.sample_list.txt', header = None, index = None)

In [ ]:
!dx upload UKBB.AD.ALL.sample_list.txt --path :/AD_GWAS/input/

# download cleaned PC files

In [ ]:
!dx download :/AD_GWAS/output/pca/ukb21007_all_chr_b0_v1.topmed_imputed.AD.PCA_cleaned.*

# read in PC files

In [ ]:
eigenval = pd.read_csv('ukb21007_all_chr_b0_v1.topmed_imputed.AD.PCA_cleaned.eigenval', header = None)
print(len(eigenval.index))
eigenval.head()

In [ ]:
eigenvec = pd.read_csv('ukb21007_all_chr_b0_v1.topmed_imputed.AD.PCA_cleaned.eigenvec', sep = '\t', header = None)
print(len(eigenvec.index))
eigenvec.head()

# make scree plot

## clean eigenval files

In [ ]:
eigenval['PC'] = list(range(1, 21))
eigenval.rename(columns = {0 : 'EIGENVAL'}, inplace = True)
eigenval['VARIANCE'] = eigenval['EIGENVAL']/eigenval['EIGENVAL'].sum()
eigenval.head()

## make plot

In [ ]:
plt.plot(eigenval['PC'], eigenval['VARIANCE'])
plt.xlabel('PC')
plt.ylabel('Variance')
plt.title('UKB AD GWAS Scree Plot')
plt.xticks(list(range(1, 21)))

# subset eigenvec file

In [ ]:
eigenvec_sub=eigenvec[[0, 1, 2, 3]]
eigenvec_sub.rename(columns = {0 : 'id', 1 : 'PC1', 2 : 'PC2', 3 : 'PC3'}, inplace = True)
eigenvec_sub.head()

# merge PCs with pheno covar

In [ ]:
case_control_pc = case_control_age_65.merge(eigenvec_sub, on = 'id', how = 'inner')
print(len(case_control_pc.index))
print(case_control_pc['AD'].value_counts())
print(case_control_pc[['AD', 'SEX']].value_counts())
case_control_pc.head()

# export pheno covar

In [ ]:
case_control_pc.to_csv('UKB.AD.ALL.phenotype_covariates.txt', sep = '\t', index = None)

In [ ]:
!dx upload UKB.AD.ALL.phenotype_covariates.txt --path :/AD_GWAS/input/

# upload jupyter notebook

In [ ]:
!dx upload AD_GWAS_Phenotyping.ipynb --path :/AD_GWAS/scripts/